# MLflow solves this in three ways

- First - Experiment tracking. Ever model run gets logged automatically. Parameters, metrics, artifacts. Timestamped, named, searchable. You never lose a result again.
- Second - Model registry. Your best model gets registered with a version number. Verson 1 goes to staging. After validation it gets promoted to production. Your serving layer always knows which version is live.
- Third - Reproducibility. Any run can be re-executed exactly. Same parameters, same data version, same results. That's the production guarantee.

In [1]:
# WITHOUT MLFlow                    WITH MLFlow
#---------------                    ------------
# print("accuracy: 0.95")           mlflow.log_metric("accuracy", 0.95)
# where did I save this?            -> stored, searchable, versioned
# which params did I use?           -> params logged automatically
# which was the best run?           -> UI comparison in one click
# can I reproduce run 3?            -> yes, run ID captures everything


# Phase 4 - MLflow Experiment Tracking
## Healthcare AI System

**Prerequisities**
- MLflow UI running at http://localhost:5000
- `model_table.csv` in outputs folder

**Run in the terminal before this notebook:**
```bash
cd AISysMlOps
mlflow ui --port 5000
```

In [39]:
import warnings

import mlflow
import os
import joblib
import pandas as pd
import json
warnings.filterwarnings("ignore")
from sklearn.metrics import (accuracy_score, f1_score, recall_score)

# point MLflow to project root
mlflow.set_tracking_uri(f"sqlite:///../mlflow.db")
print("MLflow version: ", mlflow.__version__)
print("Tracking URI: ", mlflow.get_tracking_uri())

MLflow version:  3.15.1
Tracking URI:  sqlite:///../mlflow.db


In [40]:
# Set the Experiment
mlflow.set_experiment("healthcare-risk-classification")
print("Experiment Created")

Experiment Created


In [41]:
# Step 1 Load the saved models
risk_rf_model = joblib.load("../models/risk_model.joblib")
claim_rf_model = joblib.load("../models/claim_model.joblib")
print("Models Loaded")
print("risk_model", type(risk_rf_model))
print("claim_model", type(claim_rf_model))

Models Loaded
risk_model <class 'sklearn.pipeline.Pipeline'>
claim_model <class 'sklearn.pipeline.Pipeline'>


In [42]:
# Step 2 Load the dataset
df = pd.read_csv("../outputs/model_table.csv", parse_dates=["registration_date", "visit_date", "billing_date"])
print("Dataset Loaded")
print("Shape of dataset:", df.shape)
df.head()

Dataset Loaded
Shape of dataset: (25000, 30)


,patient_id,age,gender,city,insurance_provider,chronic_flag,registration_date,visit_id,visit_date,department,...,risk_numeric,claim_numeric,is_rejected,days_since_registration,visit_frequency,avg_los_per_patient,provider_rejection_rate,visit_month,visit_dayofweek,high_cost_visit_flag
0,2,15,F,Mumbai,CareOne,0,2025-12-27,8,2026-01-01,General,...,0,2,1,5,4,21.120000,0.256876,1,3,0
1,12,3,M,Bangalore,CareOne,0,2025-08-13,65,2026-01-01,ICU,...,2,2,1,141,8,23.750000,0.256876,1,3,1
2,129,44,M,Pune,MediCareX,1,2025-07-20,651,2026-01-01,ICU,...,2,1,0,165,3,32.460000,0.242556,1,3,1
3,133,47,F,Delhi,CareOne,1,2025-11-02,670,2026-01-01,General,...,1,0,0,60,3,30.056667,0.256876,1,3,0
4,139,14,F,Chennai,SecureLife,1,2025-02-05,706,2026-01-01,Cardiology,...,1,0,0,330,9,29.030000,0.157496,1,3,1


In [43]:
# Step 3 Load feature schema
with open("../outputs/feature_schema.json", "r") as f:
    schema = json.load(f)

risk_features = schema["risk_model_features"]
claim_features = schema["claim_model_features"]
risk_target = schema["risk_target"]
claim_target = schema["claim_target"]

print("Schema Loaded")
print(f"Risk features: {len(risk_features)}")
print(f"Claim features: {len(claim_features)}")

Schema Loaded
Risk features: 14
Claim features: 18


In [44]:
# Step 4 Create separate datasets
risk_df = df.copy()
claim_df = df.copy()

In [45]:
# Step 5 Time based split
risk_df = risk_df.sort_values("visit_date").reset_index(drop=True)
split_idx = int(len(risk_df) * 0.8)

risk_train = risk_df.iloc[:split_idx].copy()
risk_test = risk_df.iloc[split_idx:].copy()

X_train_risk = risk_train[risk_features]
X_test_risk = risk_test[risk_features]

y_train_risk = risk_train[risk_target]
y_test_risk = risk_test[risk_target]

print("Risk Train shape:", X_train_risk.shape)
print("Risk Test shape:", X_test_risk.shape)
print("Risk train period: ", risk_train["visit_date"].min().date(), " - ", risk_train["visit_date"].max().date())
print("Risk test period: ", risk_test["visit_date"].min().date(), " - ", risk_test["visit_date"].max().date())

Risk Train shape: (20000, 14)
Risk Test shape: (5000, 14)
Risk train period:  2025-01-21  -  2026-01-02
Risk test period:  2026-01-02  -  2026-01-20


In [46]:
# Step 6 Time Based Split - for Claim
claim_df = claim_df.sort_values("billing_date").reset_index(drop=True)
split_idx = int(len(claim_df) * 0.8)

claim_train = claim_df.iloc[:split_idx].copy()
claim_test = claim_df.iloc[split_idx:].copy()

X_train_claim = claim_train[claim_features]
X_test_claim = claim_test[claim_features]

y_train_claim = claim_train[claim_target]
y_test_claim = claim_test[claim_target]

print("Claim Train shape:", X_train_claim.shape)
print("Claim Test shape:", X_test_claim.shape)
print("Claim train period: ", claim_train["billing_date"].min().date(), " - ", claim_train["billing_date"].max().date())
print("Claim test period: ", claim_test["billing_date"].min().date(), " - ", claim_test["billing_date"].max().date())

Claim Train shape: (20000, 18)
Claim Test shape: (5000, 18)
Claim train period:  2025-01-28  -  2026-01-17
Claim test period:  2026-01-17  -  2026-01-20


In [47]:
# Step 7 Log Risk Model
with mlflow.start_run(run_name="RandomForest-Risk") as run:
    mlflow.log_params({
        "model": "RandomForestClassifier",
        "n_estimators": 200,
        "max_depth": 8,
        "min_samples_split": 20,
        "min_samples_leaf": 10,
        "class_weight": "balanced_subsample",
        "evaluation_data": "risk_test_only",
        "split_strategy": "time_based_80_20",
        "split_column": "visit_date"
    })

    pred_risk = risk_rf_model.predict(X_test_risk)

    # Metrics
    acc_risk = accuracy_score(y_test_risk, pred_risk)
    f1_risk = f1_score(y_test_risk, pred_risk, average="weighted")
    high_recall = recall_score(y_test_risk, pred_risk, labels=["High"], average=None)[0]

    # Logging the Metrics
    mlflow.log_metric("accuracy", acc_risk)
    mlflow.log_metric("weighted_f1", f1_risk)
    mlflow.log_metric("high_risk_recall", high_recall)

    # Logging the Model
    mlflow.sklearn.log_model(
        sk_model=risk_rf_model,
        name="model",
        skops_trusted_types=["numpy.dtype"]
    )

    # Fetching the run id
    risk_run_id = run.info.run_id

    print("RandomForest Risk Model Logged")
    print(f"Run ID              : {risk_run_id}")
    print(f"Accuracy            : {acc_risk:.4f}")
    print(f"Weighted F1         : {f1_risk:.4f}")
    print(f"High Risk Recall    : {high_recall:.4f}")


RandomForest Risk Model Logged
Run ID              : fc9d80e5706c47d58dadd2d49a094f54
Accuracy            : 0.9350
Weighted F1         : 0.9350
High Risk Recall    : 0.9296


In [48]:
# Step 8 Log Claim Model
with mlflow.start_run(run_name="RandomForest-Claim") as run:

    mlflow.log_params({
        "model": "RandomForestClassifier",
        "n_estimators": 250,
        "max_depth": 14,
        "min_samples_split": 8,
        "class_weight": "balanced",
        "evaluation_data": "claim_test_only",
        "split_strategy": "time_based_80_20",
        "split_column": "billing_date"
    })

    pred_claim = claim_rf_model.predict(X_test_claim)

    acc_claim = accuracy_score(y_test_claim, pred_claim)
    f1_claim = f1_score(y_test_claim, pred_claim, average="weighted")
    rejected_recall = recall_score(
        y_test_claim, pred_claim, labels=["Rejected"], average=None
    )[0]

    mlflow.log_metric("accuracy", acc_claim)
    mlflow.log_metric("weighted_f1", f1_claim)
    mlflow.log_metric("rejected_recall", rejected_recall)

    mlflow.sklearn.log_model(
        sk_model=claim_rf_model,
        name="model",
        skops_trusted_types=["numpy.dtype"]
    )

    claim_run_id = run.info.run_id

    print("RandomForest Claim Model Logged")
    print(f"Run ID              : {claim_run_id}")
    print(f"Accuracy            : {acc_claim:.4f}")
    print(f"Weighted F1         : {f1_claim:.4f}")
    print(f"Rejected Recall     : {rejected_recall:.4f}")

RandomForest Claim Model Logged
Run ID              : 14a7654654b1490dbd34181f737fa2f1
Accuracy            : 0.4242
Weighted F1         : 0.4175
Rejected Recall     : 0.2048


In [49]:
# Step 9 Print the final summary
print("="*60)
print("MLFLOW RUNS SUMMARY")
print("="*60)
print(f"Claim Model Run ID  : {claim_run_id}")
print(f"Risk Model Run ID   : {risk_run_id}")
print()
print("Open MLflow UI at: http://localhost:5000")

MLFLOW RUNS SUMMARY
Claim Model Run ID  : 14a7654654b1490dbd34181f737fa2f1
Risk Model Run ID   : fc9d80e5706c47d58dadd2d49a094f54

Open MLflow UI at: http://localhost:5000


## Register the Model
Registration creates a named, versioned entry in the MLflow Model Registry.
Every time you register the same names, the version number increments automatically. v1 -> v2 -> v3 and so on.

In [50]:
# Step 1 Register the RIsk RF Model in Model Registry
from mlflow import register_model

register_model_name = "HealthRiskRFModel"
model_uri = f"runs:/{risk_run_id}/model"

result = register_model(
    model_uri=model_uri,
    name=register_model_name
)

print("Register Model Name: ", result.name)
print("Registered version: ", result.version)

Successfully registered model 'HealthRiskRFModel'.
2026/09/03 08:51:12 WARNING mlflow.tracking._model_registry.fluent: Run with id fc9d80e5706c47d58dadd2d49a094f54 has no artifacts at artifact path 'model', registering model based on models:/m-a4a6324886054c47bcd783600387796d instead


Register Model Name:  HealthRiskRFModel
Registered version:  1


Created version '1' of model 'HealthRiskRFModel'.


In [51]:
# Step 2 Saving the Registered version in a variable
risk_model_version = result.version
print(f"Risk Model Version Saved: {risk_model_version}")

Risk Model Version Saved: 1


In [52]:
# Step 3 Create MLflow Client
from mlflow.tracking import MlflowClient
client = MlflowClient()
print("MLflow Client Ready")

MLflow Client Ready


In [53]:
# Step 4 Move Risk Model version 1 to Staging
client.transition_model_version_stage(
    name=register_model_name,
    version=risk_model_version,
    stage="Staging"
)
print(f"Model {register_model_name} Version {risk_model_version} moved to Staging")

Model HealthRiskRFModel Version 1 moved to Staging


In [54]:
# Step 4 Check if model qualifies for Production
if acc_risk >= 0.55 and high_recall >= 0.70:
    print("Risk Model is eligible for production promotion")
else:
    print("Risk Model is not eligible")

Risk Model is eligible for production promotion


In [55]:
# Step 6 Promote the Risk Model to Production
client.transition_model_version_stage(
    name=register_model_name,
    version=risk_model_version,
    stage="Production"
)
print(f"Model {register_model_name} Version {risk_model_version} moved to Production")

Model HealthRiskRFModel Version 1 moved to Production


In [56]:
# Stap 7 Load the Production Risk Model from Registry
import mlflow.sklearn
production_risk_model = mlflow.sklearn.load_model(
    model_uri=f"models:/{register_model_name}/Production"
)
print("Production Risk Model Loaded")

Production Risk Model Loaded


In [57]:
# Step 8 Fetch the current Production version
latest_versions = client.get_latest_versions(
    name=register_model_name,
    stages=["Production"]
)

production_version = latest_versions[0].version if latest_versions else None
print(f"Current production version: {production_version}")

Current production version: 1


In [58]:
# Step 9 Run a prediction using the Production Risk Model
pred_risk = production_risk_model.predict(X_test_risk.head(5))
print("Prediction using Production Risk Model:", pred_risk)

Prediction using Production Risk Model: ['Medium' 'Medium' 'Medium' 'Low' 'Medium']


In [59]:
# Step 10 Log Prediction with Model Version
import hashlib
from datetime import datetime

def hash_input(payload: dict) -> str:
    payload_str = json.dumps(payload, sort_keys=True)
    return hashlib.sha256(payload_str.encode()).hexdigest()

In [60]:
# Step 11 Input Payload
input_payload = {
    "age": 52,
    "gender": "Male",
    "city": "Bangalore",
    "insurance_provider": "ABC Insurance",
    "chronic_flag": 1,
    "department": "Cardiology",
    "visit_type": "Emergency",
    "doctor_id": 101,
    "length_of_stay_hours": 48,
    "days_since_registration": 300,
    "visit_frequency": 4,
    "avg_los_per_patient": 36.5,
    "visit_month": 3,
    "visit_dayofweek": 2
}

In [62]:
# Step 12 Generate Hash
input_hash = hash_input(input_payload)
print(f"Input Hash: {input_hash}")

Input Hash: 957ea3a7e4ce53406465a62562a6152ea4875786e0af2d658857a2bb4d734a58


In [66]:
# Step 13 Creating a Log Record

# Define logs directory
BASE_DIR = os.getcwd()
LOG_DIR = os.path.join(BASE_DIR, "logs")
os.makedirs(LOG_DIR, exist_ok=True)

LOG_FILE = os.path.join(LOG_DIR, "predictions.log")
prediction_log = {
    "timestamp": datetime.utcnow().isoformat(),
    "model_name": register_model_name,
    "model_version": production_version,
    "input_hash": input_hash,
    "prediction": str(pred_risk[0])
}

with open(LOG_FILE, "a", encoding="utf-8") as log_file:
    log_file.write(json.dumps(prediction_log) + "\n")

print("Prediction Logged Successfully with model_version")

Prediction Logged Successfully with model_version
